# 07 · Accretion / dilution of Disney EPS

* **Disney standalone:** "Disney street forecasts" (segment operating income) and "street estimates" (corporate expense),
  from May 2018 proxy pp.152–153. These are analyst-based, approved by Disney for its advisors' use, and not prepared by
  Disney management.
* **Acquired business (RemainCo):** Disney's forecasts for RemainCo (proxy p.129), moved from June to September fiscal years.
* **Financing (closing):** $35.7bn cash, of which $16.1bn of the bridge was repaid from acquired cash; 306.8m new shares;
  3.3% on new debt (Disney's 8-K/A pro forma assumption).
* **Purchase accounting:** $2.7bn a year of incremental amortization excluding the RSNs (FY2019 10-K pro forma).
* **Synergies and integration costs:** the Stage 4A base case.
* **EPS definitions:** *adjusted* EPS excludes purchase-accounting amortization and integration costs (management's "before
  purchase accounting" definition); *GAAP basis* EPS includes both.

Disney's own reported pro forma (8-K/A, FY2018) is the anchor: $8.36 standalone against $5.94 pro forma.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 300)
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
res = subprocess.run([sys.executable, str(ROOT / "src" / "valuation" / "build_stage4.py")], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
print(res.stdout.strip()); assert res.returncode == 0, res.stdout + res.stderr
V = pd.read_csv(PROC / "verification_checks.csv")
AI = pd.read_csv(PROC / "accretion_dilution_inputs.csv"); AO = pd.read_csv(PROC / "accretion_dilution_outputs.csv"); AO["v"] = pd.to_numeric(AO.value, errors="coerce")

In [ ]:
AI

## Reported anchor and model cases

In [ ]:
AO[AO.case.str.startswith("reported")][["fiscal_year", "metric", "v", "notes"]]

In [ ]:
m = ["disney_standalone_eps", "pro_forma_eps_adjusted", "accretion_adjusted", "pro_forma_eps_gaap_basis", "accretion_gaap_basis"]
AO[AO.metric.isin(m) & ~AO.case.str.startswith(("sensitivity", "reported"))].pivot_table(index=["case", "metric"], columns="fiscal_year", values="v").round(3)

In [ ]:
b = AO[AO.case.str.startswith("base")]
b[b.fiscal_year != "timeline"].pivot_table(index="metric", columns="fiscal_year", values="v").round(2)

In [ ]:
AO[AO.fiscal_year == "timeline"][["metric", "value", "notes"]]

## Sensitivity of FY2021 accretion (the second fiscal year after the March 2019 close)

In [ ]:
s = AO[AO.case.str.startswith("sensitivity") & (AO.fiscal_year == "FY2021") & AO.metric.isin(["accretion_adjusted", "accretion_gaap_basis"])]
s.pivot_table(index="case", columns="metric", values="v").round(4)

In [ ]:
v = V[V.check_type.str.startswith("accretion_")]; print(v.groupby(["check_type", "status"]).size()); assert (v.status == "pass").all()